*Module 7 of 9*

> **¿Prefieres español?** Abre [`07_machine_learning_desde_cero.ipynb`](../es/07_machine_learning_desde_cero.ipynb) — es el mismo módulo, en español.


# 🤖 Module 7 — Machine learning from zero

🧭 **Objectives** — understand what a **model** is, meet the **decision tree**
and the **random forest**, learn the golden rule of honest evaluation
(**train/test split**), train a classifier on your labelled parcels, and
*read* the quality report: **accuracy**, **precision/recall**, **F1**,
**kappa**, and the **confusion matrix** — plus **omission** vs **commission**
errors.

📚 **What is a model?** A model is a function learned from examples. We show
it parcels whose crop we know (features → label), and it learns rules to
predict the crop of parcels it has never seen. This is **supervised
learning**.

📚 **Decision tree → random forest.** A **decision tree** asks yes/no
questions about the features ("is NIR mean > 3000?") down to a leaf that
names a crop. One tree overfits — it memorizes quirks. A **random forest**
grows hundreds of trees, each on a random slice of the data and features,
and lets them **vote**. The crowd is far more accurate and stable than any
single tree.

📚 **The golden rule.** Never judge a model on the data it trained on — of
course it aced those. Split the labelled parcels into a **training set** (it
learns from) and a **test set** (held back, used only to grade). Accuracy on
the *test* set estimates how it will do on the real, unseen map.

![training](../../anim/en/08_training.svg)


## Rebuild features and labels

As in Module 6, a fresh kernel means we re-create the parcels, their
features, and the pure labels before training.


In [ ]:
# Get the workshop tile (a few MB; cached after the first download)
import os, sys

async def get_file(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await get_file("crop_tile_384.tif")
print("Tile ready:", TILE)

In [ ]:
import numpy as np, rasterio, json
import scipy.ndimage, sklearn.cluster
import shepherd_wasm

async def get_file(name):
    import os, sys
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url); open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request; urllib.request.urlretrieve(url, dest)
    return dest

with rasterio.open(TILE) as src:
    img = src.read(); band_names = list(src.descriptions)
LABELS = await get_file("crop_labels_384.tif")
NAMES  = await get_file("class_names.json")

result = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=30, minSegmentSize=50, imgNullVal=0, fixedKMeansInit=True)
seg = result.segimg.astype(np.int32); n_seg = int(seg.max())

flat = seg.ravel(); counts = np.bincount(flat, minlength=n_seg+1).astype(float)
counts[counts == 0] = 1
features = np.zeros((n_seg+1, len(band_names)*2), dtype=np.float32)
for b in range(len(band_names)):
    v = img[b].ravel().astype(np.float64)
    s1 = np.bincount(flat, weights=v, minlength=n_seg+1)
    s2 = np.bincount(flat, weights=v*v, minlength=n_seg+1)
    m = s1/counts; var = np.maximum(s2/counts - m*m, 0)
    features[:, 2*b], features[:, 2*b+1] = m, np.sqrt(var)

with rasterio.open(LABELS) as src:
    lab = src.read(1)
class_names = {int(k): v for k, v in json.load(open(NAMES)).items()}
parcel_label = np.zeros(n_seg+1, dtype=int)
for sid in np.unique(seg[lab > 0]):
    u = np.unique(lab[(seg == sid) & (lab > 0)])
    if len(u) == 1: parcel_label[sid] = u[0]
train_ids = np.flatnonzero(parcel_label)
print(f"Ready: {len(train_ids)} labelled parcels, {features.shape[1]} features each")

## Split, train, and grade honestly

`train_test_split` holds back 30% of the parcels for testing, **stratified**
so every crop is represented in both parts. We fit a `RandomForestClassifier`
on the training 70%, then grade it on the untouched 30% with a
`classification_report`.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = features[train_ids]
y = parcel_label[train_ids]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)

present = sorted(np.unique(y_te))
print(classification_report(
    y_te, model.predict(X_te),
    labels=present, target_names=[class_names[c] for c in present]))

## Read the report

- **precision** of a crop = of the parcels the model *called* this crop, how
  many really were? Low precision = **commission** error (false alarms).
- **recall** of a crop = of the parcels that *truly* were this crop, how many
  did the model catch? Low recall = **omission** error (misses).
- **f1-score** = the balance of precision and recall in one number.
- **accuracy** (bottom) = overall fraction correct across all crops.

Now visualize the **confusion matrix**: rows = true crop, columns =
predicted. The diagonal is correct; off-diagonal cells show *which* crops get
confused for which — far more informative than a single accuracy number.


In [ ]:
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import matplotlib.pyplot as plt

y_pred = model.predict(X_te)
cm = confusion_matrix(y_te, y_pred, labels=present)
print("Cohen's kappa (agreement beyond chance):", round(cohen_kappa_score(y_te, y_pred), 3))

fig, ax = plt.subplots(figsize=(6.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(present))); ax.set_yticks(range(len(present)))
ax.set_xticklabels([class_names[c] for c in present], rotation=45, ha="right")
ax.set_yticklabels([class_names[c] for c in present])
ax.set_xlabel("predicted"); ax.set_ylabel("true")
for i in range(len(present)):
    for j in range(len(present)):
        ax.text(j, i, cm[i, j], ha="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_title("Confusion matrix (diagonal = correct)")
plt.tight_layout(); plt.show()

## 🧪 Check yourself

**Why is accuracy measured on the test set, not the training set?**

<details><summary>Show answer</summary>

A model can memorize its training data and score near-perfectly on it while
failing on new parcels (overfitting). The held-back test set was never seen
during training, so its accuracy honestly estimates performance on the real,
unseen map.

</details>

**A crop has high precision but low recall. In plain terms, what is the
model doing — and is that omission or commission?**

<details><summary>Show answer</summary>

When it says "this crop" it is usually right (high precision), but it misses
many true parcels of that crop (low recall). Missing true positives is
**omission** error.

</details>

**Why prefer a random forest over a single decision tree?**

<details><summary>Show answer</summary>

One tree overfits — it latches onto quirks of the training data. A forest
averages hundreds of varied trees, canceling out individual mistakes, giving
more accurate and more stable predictions.

</details>


## 🔭 Go deeper

Optional: these bilingual concept cards expand what you just learned
(prerequisite chains, lineage to fundamentals, curated references):

- [Machine learning](https://abxda.github.io/rs-learning-audio/?id=machine-learning)
- [Decision tree](https://abxda.github.io/rs-learning-audio/?id=decision-tree)
- [Random forest](https://abxda.github.io/rs-learning-audio/?id=random-forest)
- [Model training](https://abxda.github.io/rs-learning-audio/?id=model-training)
- [Training vs test data](https://abxda.github.io/rs-learning-audio/?id=training-dataset)
- [Validation](https://abxda.github.io/rs-learning-audio/?id=validation)
- [Confusion matrix](https://abxda.github.io/rs-learning-audio/?id=confusion-matrix)
- [Overall accuracy](https://abxda.github.io/rs-learning-audio/?id=overall-accuracy)
- [Producer's accuracy](https://abxda.github.io/rs-learning-audio/?id=producer-s-accuracy)
- [Recall](https://abxda.github.io/rs-learning-audio/?id=recall)
- [F1-score](https://abxda.github.io/rs-learning-audio/?id=f1-score)
- [Cohen's kappa](https://abxda.github.io/rs-learning-audio/?id=cohen-s-kappa)
- [Omission error](https://abxda.github.io/rs-learning-audio/?id=omission)
- [Commission error](https://abxda.github.io/rs-learning-audio/?id=commission)



---

[← Previous · Module 6 — Parcels become a table + the ground truth](06_features_and_labels.ipynb) · [Next → · Module 8 — Capstone: the crop map, end to end](08_capstone_crop_map.ipynb)
